In [0]:
import pandas as pd, time, os, psutil

OUT = "/Volumes/workspace/default/tenday"
proc = psutil.Process(os.getpid())

for n in [100_000, 1_000_000, 5_000_000]:
    try:
        t0 = time.time()
        m0 = proc.memory_info().rss / 1e9
        pdf = (spark.read.format("delta").load(f"{OUT}/transactions")
                    .limit(n).toPandas())
        pdf = pdf.dropna(subset=["amount"])
        pdf = pdf[pdf.amount > 0]
        pdf["cum"] = pdf.sort_values("days_since_acquisition") \
                        .groupby("customer_id")["amount"].cumsum()
        agg = pdf.groupby("customer_id").agg(
            total=("amount", "sum"), cnt=("amount", "count"),
            cats=("merchant_category", "nunique"))
        peak = proc.memory_info().rss / 1e9
        print(f"{n:>10,} rows | {time.time()-t0:6.1f}s | ~{peak-m0:.2f} GB")
    except Exception as e:
        print(f"{n:>10,} rows | FAILED: {type(e).__name__}")
        break

   100,000 rows |    1.7s | ~0.08 GB
 1,000,000 rows |    4.1s | ~0.26 GB
 5,000,000 rows |   17.6s | ~1.13 GB


In [0]:
"""
Tenday -- Notebook 02: cleaning + horizon-separated feature engineering.

Produces four tables:

    txns_clean      deduplicated, typed, refunds separated
    features_l1     acquisition-time only      (no behaviour at all)
    features_l2     first EARLY_WINDOW days    (early behaviour)
    features_l3     full 365-day window        (outcome + full behaviour)

THE GOVERNING RULE
------------------
Each layer may only use information available at its horizon. L1 sees no
transactions. L2 sees only days 0..EARLY_WINDOW. L3 sees everything.
Any leak across those boundaries invalidates the three-horizon comparison,
and that comparison is the project.
"""

from pyspark.sql import functions as F, Window

EARLY_WINDOW = 30       # days of behaviour visible to Layer 2
HORIZON_DAYS = 365
MARGIN_RATE = 0.02      # interchange margin on spend -- revenue proxy


# ----------------------------------------------------------------------
# 1. CLEANING
# ----------------------------------------------------------------------

def clean_transactions(txns):
    """
    Four decisions, each defensible:

    dedup        -- 689 exact duplicates on transaction_id (double-posted).
                    Keep one of each.
    null amount  -- ~0.4% of rows. DROPPED, not imputed. Imputing spend
                    would corrupt cumulative-spend and velocity features,
                    which are the backbone of gaming detection.
    negatives    -- refunds, not errors. SEPARATED rather than dropped:
                    excluded from spend aggregates, but counted, because a
                    refund rate is itself a behavioural signal.
    null category-- kept as 'unknown'. Dropping the row would understate
                    that customer's spend; the amount is perfectly valid.
    """
    clean = (
        txns
        .dropDuplicates(["transaction_id"])
        .filter(F.col("amount").isNotNull())
        .withColumn("merchant_category",
                    F.coalesce(F.col("merchant_category"), F.lit("unknown")))
        .withColumn("is_refund", F.col("amount") < 0)
        .withColumn("abs_amount", F.abs(F.col("amount")))
    )
    return clean


# ----------------------------------------------------------------------
# 2. LAYER 1 -- acquisition-time features
# ----------------------------------------------------------------------

def build_l1(customers, campaigns):
    """
    Everything knowable BEFORE the customer transacts even once.
    Deliberately information-poor. Measuring how poor is the deliverable.
    """
    return (
        customers
        .join(F.broadcast(campaigns), "campaign_id", "left")
        .select(
            "customer_id",
            "acquisition_date", "acquisition_month",
            "acquisition_channel", "campaign_id", "campaign_type",
            "offer_type", "age_band", "income_band", "region",
            "credit_limit", "bonus_amount", "minimum_spend",
            "acquisition_cost",
            # derived, still acquisition-time only
            (F.col("bonus_amount") / F.col("minimum_spend"))
                .alias("offer_richness"),
            (F.col("minimum_spend") / F.col("credit_limit"))
                .alias("threshold_to_limit_ratio"),
        )
    )


# ----------------------------------------------------------------------
# 3. BEHAVIOURAL FEATURES -- parameterised by window
# ----------------------------------------------------------------------

def behavioural_features(clean_txns, window_days, prefix):
    """
    Aggregate transactions to customer level over days 0..window_days.

    The same function builds L2 (30 days) and the behavioural part of L3
    (365 days). Writing it once guarantees the two horizons are computed
    identically -- only the window differs. That is what makes the
    comparison between them meaningful.
    """
    w = clean_txns.filter(F.col("days_since_acquisition") < window_days)

    spend = w.filter(~F.col("is_refund"))

    # --- volume / value -------------------------------------------------
    base = spend.groupBy("customer_id").agg(
        F.count("*").alias(f"{prefix}_txn_count"),
        F.sum("abs_amount").alias(f"{prefix}_total_spend"),
        F.avg("abs_amount").alias(f"{prefix}_avg_txn"),
        F.expr("percentile_approx(abs_amount, 0.5)").alias(f"{prefix}_median_txn"),
        F.max("abs_amount").alias(f"{prefix}_max_txn"),
        F.stddev("abs_amount").alias(f"{prefix}_std_txn"),
        F.min("days_since_acquisition").alias(f"{prefix}_first_day"),
        F.max("days_since_acquisition").alias(f"{prefix}_last_day"),
        F.countDistinct("days_since_acquisition").alias(f"{prefix}_active_days"),
        F.countDistinct("merchant_id").alias(f"{prefix}_distinct_merchants"),
        F.countDistinct("merchant_category").alias(f"{prefix}_distinct_categories"),
    )

    # --- refund behaviour -----------------------------------------------
    refunds = w.groupBy("customer_id").agg(
        F.sum(F.col("is_refund").cast("int")).alias(f"{prefix}_refund_count")
    )

    # --- category concentration (Herfindahl index) ----------------------
    # HHI = sum of squared category shares. 1.0 = all spend in one
    # category; ~1/13 = perfectly spread. Gamers concentrate.
    cat = (spend.groupBy("customer_id", "merchant_category")
                .agg(F.sum("abs_amount").alias("cat_spend")))
    tot = (cat.groupBy("customer_id")
              .agg(F.sum("cat_spend").alias("tot_spend")))
    hhi = (cat.join(tot, "customer_id")
              .withColumn("share", F.col("cat_spend") / F.col("tot_spend"))
              .groupBy("customer_id")
              .agg(F.sum(F.col("share") ** 2).alias(f"{prefix}_category_hhi")))

    # --- liquid-category share ------------------------------------------
    # Gift cards, money transfer and wholesale clubs are cash-equivalent.
    # Heavy use is the classic manufactured-spend pattern.
    liquid = (
        spend
        .withColumn("is_liquid",
                    F.col("merchant_category").isin(
                        "gift_cards", "money_transfer", "wholesale_club"))
        .groupBy("customer_id")
        .agg((F.sum(F.when(F.col("is_liquid"), F.col("abs_amount"))
                     .otherwise(0.0))
              / F.sum("abs_amount")).alias(f"{prefix}_liquid_share"))
    )

    # --- channel mix ------------------------------------------------------
    chan = (
        spend.groupBy("customer_id")
        .agg((F.sum(F.when(F.col("channel") == "online", F.col("abs_amount"))
                     .otherwise(0.0)) / F.sum("abs_amount"))
             .alias(f"{prefix}_online_share"),
             (F.sum(F.when(F.col("channel") == "recurring", F.col("abs_amount"))
                     .otherwise(0.0)) / F.sum("abs_amount"))
             .alias(f"{prefix}_recurring_share"))
    )

    out = (base
           .join(refunds, "customer_id", "left")
           .join(hhi, "customer_id", "left")
           .join(liquid, "customer_id", "left")
           .join(chan, "customer_id", "left"))

    # --- derived rates ----------------------------------------------------
    out = (
        out
        .withColumn(f"{prefix}_spend_velocity",
                    F.col(f"{prefix}_total_spend") / F.lit(window_days))
        .withColumn(f"{prefix}_txn_frequency",
                    F.col(f"{prefix}_txn_count") / F.lit(window_days))
        .withColumn(f"{prefix}_active_day_ratio",
                    F.col(f"{prefix}_active_days") / F.lit(window_days))
        .withColumn(f"{prefix}_dormancy_days",
                    F.lit(window_days - 1) - F.col(f"{prefix}_last_day"))
        .withColumn(f"{prefix}_spend_per_active_day",
                    F.col(f"{prefix}_total_spend")
                    / F.greatest(F.lit(1), F.col(f"{prefix}_active_days")))
    )
    return out


def spend_trajectory(clean_txns, prefix="full"):
    """
    Front-loading ratio: share of full-window spend that happened in the
    first 30 days. A gamer's spend is nearly all up front; a steady
    builder's is spread. Full-window feature -- Layer 3 only.
    """
    spend = clean_txns.filter(~F.col("is_refund"))
    return (
        spend.groupBy("customer_id").agg(
            (F.sum(F.when(F.col("days_since_acquisition") < 30,
                          F.col("abs_amount")).otherwise(0.0))
             / F.sum("abs_amount")).alias(f"{prefix}_first30_spend_share"),
            (F.sum(F.when(F.col("days_since_acquisition") >= 180,
                          F.col("abs_amount")).otherwise(0.0))
             / F.sum("abs_amount")).alias(f"{prefix}_last6mo_spend_share"),
        )
    )


# ----------------------------------------------------------------------
# 4. REDEMPTION FEATURES  (horizon-aware)
# ----------------------------------------------------------------------

def redemption_features(redemptions, window_days, prefix):
    """
    Qualification and redemption are events with dates, so they must be
    censored to the horizon like everything else. A customer who qualifies
    on day 200 has NOT qualified as far as Layer 2 is concerned.
    """
    r = redemptions
    qual_in = F.col("days_to_qualify") < window_days
    redeem_day = F.col("days_to_qualify") + F.col("redemption_lag_days")
    redeem_in = redeem_day < window_days

    return r.select(
        "customer_id",
        F.when(qual_in, F.lit(1)).otherwise(F.lit(0))
            .alias(f"{prefix}_qualified"),
        F.when(qual_in, F.col("days_to_qualify"))
            .alias(f"{prefix}_days_to_qualify"),
        F.when(F.col("bonus_redeemed") & redeem_in, F.lit(1)).otherwise(F.lit(0))
            .alias(f"{prefix}_redeemed"),
        F.when(F.col("bonus_redeemed") & redeem_in,
               F.col("redemption_lag_days"))
            .alias(f"{prefix}_redemption_lag"),
        F.when(F.col("bonus_redeemed") & redeem_in, F.col("bonus_earned"))
            .otherwise(F.lit(0)).alias(f"{prefix}_bonus_earned"),
    )


# ----------------------------------------------------------------------
# 5. OUTCOME  (Layer 3 target)
# ----------------------------------------------------------------------

def customer_outcome(l3_behaviour, l3_redemption, l1):
    """
    net_value = spend * margin - bonus paid - acquisition cost

    This is the number the business actually cares about, and the target
    the Layer 2 model tries to predict from 30 days of behaviour.
    """
    return (
        l3_behaviour.select("customer_id", "full_total_spend")
        .join(l3_redemption.select("customer_id", "full_bonus_earned"),
              "customer_id", "left")
        .join(l1.select("customer_id", "acquisition_cost"),
              "customer_id", "left")
        .withColumn("gross_revenue",
                    F.col("full_total_spend") * F.lit(MARGIN_RATE))
        .withColumn("net_value",
                    F.col("gross_revenue")
                    - F.coalesce(F.col("full_bonus_earned"), F.lit(0))
                    - F.coalesce(F.col("acquisition_cost"), F.lit(0)))
        .withColumn("is_profitable", (F.col("net_value") > 0).cast("int"))
        .select("customer_id", "gross_revenue", "net_value", "is_profitable")
    )


# ----------------------------------------------------------------------
# 6. ORCHESTRATION
# ----------------------------------------------------------------------

def build_all(spark, out, write_format="delta"):
    txns  = spark.read.format(write_format).load(f"{out}/transactions")
    cust  = spark.read.format(write_format).load(f"{out}/customers")
    camp  = spark.read.format(write_format).load(f"{out}/campaigns")
    red   = spark.read.format(write_format).load(f"{out}/redemptions")

    clean = clean_transactions(txns)
    clean.write.format(write_format).mode("overwrite").save(f"{out}/txns_clean")
    clean = spark.read.format(write_format).load(f"{out}/txns_clean")

    # ---- Layer 1 --------------------------------------------------------
    l1 = build_l1(cust, camp)
    l1.write.format(write_format).mode("overwrite").save(f"{out}/features_l1")
    l1 = spark.read.format(write_format).load(f"{out}/features_l1")

    # ---- Layer 2: acquisition-time + first 30 days -----------------------
    l2 = (l1
          .join(behavioural_features(clean, EARLY_WINDOW, "d30"),
                "customer_id", "left")
          .join(redemption_features(red, EARLY_WINDOW, "d30"),
                "customer_id", "left"))
    l2.write.format(write_format).mode("overwrite").save(f"{out}/features_l2")

    # ---- Layer 3: full window -------------------------------------------
    l3_beh = behavioural_features(clean, HORIZON_DAYS, "full")
    l3_red = redemption_features(red, HORIZON_DAYS, "full")
    outcome = customer_outcome(l3_beh, l3_red, l1)

    l3 = (l1
          .join(l3_beh, "customer_id", "left")
          .join(l3_red, "customer_id", "left")
          .join(spend_trajectory(clean), "customer_id", "left")
          .join(outcome, "customer_id", "left"))
    l3.write.format(write_format).mode("overwrite").save(f"{out}/features_l3")

    for t in ["txns_clean", "features_l1", "features_l2", "features_l3"]:
        df = spark.read.format(write_format).load(f"{out}/{t}")
        print(f"{t:<14} rows={df.count():>9,}  cols={len(df.columns)}")

In [0]:
import time
OUT = "/Volumes/workspace/default/tenday"

t0 = time.time()
build_all(spark, OUT, "delta")
print(f"\nElapsed: {time.time()-t0:.1f}s")

txns_clean     rows=26,523,823  cols=11
features_l1    rows=  200,000  cols=16
features_l2    rows=  200,000  cols=42
features_l3    rows=  200,000  cols=47

Elapsed: 35.8s


In [0]:
from pyspark.sql import functions as F

l2 = spark.read.format("delta").load(f"{OUT}/features_l2")
l3 = spark.read.format("delta").load(f"{OUT}/features_l3")
lab = spark.read.format("delta").load(f"{OUT}/hidden_labels")

# no full-window column should appear in L2
print("leaks in L2:", [c for c in l2.columns if c.startswith("full_")])

l3.join(lab, "customer_id").groupBy("true_archetype").agg(
    F.round(F.avg("full_category_hhi"), 3).alias("hhi"),
    F.round(F.avg("full_liquid_share"), 3).alias("liquid"),
    F.round(F.avg("full_days_to_qualify"), 1).alias("days_to_qual"),
    F.round(F.avg("full_redemption_lag"), 1).alias("redeem_lag"),
    F.round(F.avg("net_value"), 0).alias("net_value"),
    F.count("*").alias("n"),
).orderBy("net_value").show()

leaks in L2: []
+--------------+-----+------+------------+----------+---------+-----+
|true_archetype|  hhi|liquid|days_to_qual|redeem_lag|net_value|    n|
+--------------+-----+------+------------+----------+---------+-----+
|   bonus_gamer|0.509|  0.48|         9.8|       2.5|  -2656.0|28490|
|steady_builder| 0.16|  0.23|       185.4|      45.6|   -978.0|66523|
|    occasional|0.266|  0.23|       316.8|      37.2|   -768.0|46814|
| early_churner|0.297| 0.232|        96.5|      46.2|   -746.0|31166|
|    high_value| 0.12|  0.23|       108.4|      46.4|  11989.0|27007|
+--------------+-----+------+------------+----------+---------+-----+

